# Linear SVM

A soft-margin SVM finds a separating hyperplane while penalizing margin violations. For labels $y_i\in\{-1,1\}$, the primal problem is

$$\min_{w,b,\xi}\;\frac12\|w\|_2^2+C\sum_i\xi_i,\qquad y_i(w^\top x_i+b)\ge 1-\xi_i,\quad \xi_i\ge0.$$

Its dual maximizes $\sum_i\alpha_i-\frac12\sum_{i,j}\alpha_i\alpha_j y_i y_j x_i^\top x_j$ subject to $0\le\alpha_i\le C$ and $\sum_i y_i\alpha_i=0$.


## Implementation

The Python standard-library solver updates pairs of dual coefficients while maintaining the equality constraint and box bounds. The returned boundary stores the support vectors, signed coefficients, intercept and convergence diagnostics.


In [ ]:
from array import array
from dataclasses import dataclass
from math import exp, isfinite


@dataclass(frozen=True)
class Geometry:
    form: str = 'radial'
    radial_width: float = 1.0
    polynomial_order: int = 2

    def __post_init__(self):
        if self.form not in ('linear', 'radial', 'polynomial'):
            raise ValueError('Unknown kernel geometry.')
        if not isfinite(self.radial_width) or self.radial_width <= 0:
            raise ValueError('The radial width must be positive and finite.')
        if not isinstance(self.polynomial_order, int) or self.polynomial_order < 1:
            raise ValueError('Polynomial order must be a positive integer.')


def _checked_rows(observations, outcomes):
    records = tuple(tuple(float(v) for v in row) for row in observations)
    if not records or len(records) != len(outcomes) or not records[0]:
        raise ValueError('Nonempty matched observations/outcomes required.')
    width = len(records[0])
    if any(len(row) != width for row in records):
        raise ValueError('Nonuniform feature dimensions.')
    if any(not isfinite(v) for row in records for v in row):
        raise ValueError('Nonfinite input features.')
    return records


def _compare_rows(a, b, geometry):
    if geometry.form == 'radial':
        squared_distance = sum((u - v) * (u - v) for u, v in zip(a, b))
        return exp(-squared_distance / (2 * geometry.radial_width ** 2))
    linear_product = sum(u * v for u, v in zip(a, b))
    if geometry.form == 'linear':
        return linear_product
    return (1 + linear_product) ** geometry.polynomial_order


def _packed_kernel(records, geometry):
    """Symmetric row-major Gram table in 64-bit floating-point storage."""
    n = len(records)
    table = array('d', [0.0]) * (n * n)
    for left in range(n):
        base = left * n
        for right in range(left, n):
            value = _compare_rows(records[left], records[right], geometry)
            if not isfinite(value):
                raise FloatingPointError('Nonfinite kernel entry.')
            table[base + right] = value
            table[right * n + left] = value
    return table


@dataclass(frozen=True)
class BinaryBoundary:
    labels: tuple
    retained_examples: tuple
    signed_weights: tuple
    offset: float
    geometry: Geometry
    iterations: int
    stationarity_error: float
    dual_value: float
    certified: bool


def _binary_kkt_error(duals, orientation, scores, penalty, zero_tolerance):
    worst = 0.0
    for amount, sign, score in zip(duals, orientation, scores):
        margin = sign * score
        if amount <= zero_tolerance:
            err = max(0.0, 1 - margin)
        elif amount >= penalty - zero_tolerance:
            err = max(0.0, margin - 1)
        else:
            err = abs(margin - 1)
        if err > worst:
            worst = err
    return worst


def build_two_class_boundary(observations, outcomes, *,
                             geometry=Geometry('linear'),
                             penalty=1.0, tolerance=1e-3,
                             rounds=30000, idle_rounds=20):
    """Solve the box-constrained binary hinge-loss dual via feasible pair moves.

    Returned boundary stores only support records and dual signed weights;
    application requires evaluate_binary_boundary / label_binary_cases.
    """
    raw_labels = tuple(outcomes)
    records = _checked_rows(observations, raw_labels)
    distinct = tuple(sorted(set(raw_labels)))
    if len(distinct) != 2:
        raise ValueError('Exactly two outcome categories are necessary.')
    if not isfinite(penalty) or penalty <= 0 or tolerance <= 0:
        raise ValueError('Penalty and precision must be finite and positive.')
    if rounds < 1 or idle_rounds < 1:
        raise ValueError('Positive iteration budgets required.')

    signs = tuple(-1.0 if v == distinct[0] else 1.0 for v in raw_labels)
    kernel = _packed_kernel(records, geometry)
    n = len(records)
    multipliers = [0.0] * n
    evaluations = [0.0] * n  # includes intercept
    intercept = 0.0
    tiny = 1e-8
    idle = 0
    epoch = 0

    while epoch < rounds and idle < idle_rounds:
        changed = False
        for first in range(n):
            margin = signs[first] * evaluations[first]
            if not ((margin < 1 - tolerance and multipliers[first] < penalty - tiny) or
                    (margin > 1 + tolerance and multipliers[first] > tiny)):
                continue
            err_first = evaluations[first] - signs[first]
            partner_order = sorted(
                (other for other in range(n) if other != first),
                key=lambda other: abs(err_first - (evaluations[other] - signs[other])),
                reverse=True)
            row = first * n
            for second in partner_order:
                before_a, before_b = multipliers[first], multipliers[second]
                if signs[first] == signs[second]:
                    total = before_a + before_b
                    lo, hi = max(0.0, total - penalty), min(penalty, total)
                else:
                    difference = before_b - before_a
                    lo, hi = max(0.0, difference), min(penalty, penalty + difference)
                if hi - lo <= tiny:
                    continue
                err_second = evaluations[second] - signs[second]
                k_aa, k_bb, k_ab = kernel[row + first], kernel[second * n + second], kernel[row + second]
                curvature = max(0.0, k_aa + k_bb - 2 * k_ab)
                if curvature < 1e-13:
                    direction = signs[second] * (err_first - err_second)
                    if abs(direction) < 1e-13:
                        continue
                    new_b = hi if direction > 0 else lo
                else:
                    candidate = before_b + signs[second] * (err_first - err_second) / curvature
                    new_b = min(hi, max(lo, candidate))
                delta_b = new_b - before_b
                if abs(delta_b) < tiny * (1 + abs(new_b) + abs(before_b)):
                    continue
                new_a = before_a + signs[first] * signs[second] * (before_b - new_b)
                delta_a = new_a - before_a
                shift_a = -err_first - signs[first] * delta_a * k_aa - signs[second] * delta_b * k_ab
                shift_b = -err_second - signs[first] * delta_a * k_ab - signs[second] * delta_b * k_bb
                if tiny < new_a < penalty - tiny:
                    shift = shift_a
                elif tiny < new_b < penalty - tiny:
                    shift = shift_b
                else:
                    shift = 0.5 * (shift_a + shift_b)
                multipliers[first], multipliers[second] = new_a, new_b
                intercept += shift
                for row_index in range(n):
                    evaluations[row_index] += (
                        signs[first] * delta_a * kernel[row + row_index]
                        + signs[second] * delta_b * kernel[second * n + row_index]
                        + shift)
                changed = True
                break
        epoch += 1
        idle = 0 if changed else idle + 1

    residual = _binary_kkt_error(multipliers, signs, evaluations, penalty, tiny)
    passed = residual <= max(5 * tolerance, 1e-6) and idle >= idle_rounds
    if not passed:
        raise RuntimeError(f'Binary margin optimization not certified; residual={residual:.6g}, rounds={epoch}.')
    selected = tuple(i for i in range(n) if multipliers[i] > tiny)
    signed = tuple(multipliers[i] * signs[i] for i in selected)
    dual = sum(multipliers) - 0.5 * sum(
        multipliers[i] * signs[i] * (evaluations[i] - intercept) for i in range(n))
    return BinaryBoundary(distinct, tuple(records[i] for i in selected), signed,
                          intercept, geometry, epoch, residual, dual, True)


def evaluate_binary_boundary(boundary, observations):
    """Evaluate raw signed decision values for an existing boundary."""
    width = len(boundary.retained_examples[0]) if boundary.retained_examples else None
    values = []
    for row in observations:
        case = tuple(float(x) for x in row)
        if width is not None and len(case) != width:
            raise ValueError('Dimension mismatch in new binary observation.')
        values.append(boundary.offset + sum(
            weight * _compare_rows(stored, case, boundary.geometry)
            for stored, weight in zip(boundary.retained_examples, boundary.signed_weights)))
    return tuple(values)


def label_binary_cases(boundary, observations):
    return tuple(boundary.labels[1] if value >= 0 else boundary.labels[0]
                 for value in evaluate_binary_boundary(boundary, observations))


## Usage

Pass the training observations and labels to `build_two_class_boundary`, with `geometry=Geometry('linear')`. Use `evaluate_binary_boundary` for signed margins or `label_binary_cases` for predicted labels. Notebook 06 evaluates the classifiers on the Dermatology dataset.
